In [ ]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

In [ ]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol with an attacker, to demonstrate that the attacker can be detected.

simulator = BasicSimulator()

def quantum_random_bit():
    """
    Generate a single random bit by preparing |+> and measuring.
    This is genuine quantum randomness, not pseudo-random.
    """
    qc = QuantumCircuit(1, 1)
    qc.h(0)              # |0> -> |+> = (1/sqrt2)(|0> + |1>)
    qc.measure(0, 0)
    result = simulator.run(transpile(qc, simulator), shots=1).result()
    # Get the measured bit as an integer
    return int(list(result.get_counts().keys())[0])

def quantum_random_bits(n):
    """Generate n random bits."""
    return [quantum_random_bit() for _ in range(n)]

def quantum_random_basis(n):
    """Generate n random basis choices: 's' (standard) or 'd' (diagonal)."""
    return ['s' if quantum_random_bit() == 0 else 'd' for _ in range(n)]

In [ ]:
def eve_intercept(qc_from_alice, eve_basis):
    """
    Eve measures the intercepted qubit in her chosen basis,
    then re-prepares a fresh qubit in the state corresponding to her result
    and sends it on to Bob.
    """
    # Eve measures
    measured_bit = bob_measure(qc_from_alice, eve_basis)
    # Eve re-prepares a fresh qubit matching what she saw, in her basis
    resent_qc = alice_encode(measured_bit, eve_basis)
    return resent_qc, measured_bit


def bb84_with_attacker(n_qubits=200, sample_fraction=0.5, threshold=0.10):
    """
    Run BB84 with an intercept-resend eavesdropper.
    Use more qubits here so the statistics are clear.
    """
    print("="*60)
    print("BB84 WITH INTERCEPT-RESEND ATTACKER")
    print("="*60)

    # --- ALICE'S SIDE ---
    alice_bits  = quantum_random_bits(n_qubits)
    alice_bases = quantum_random_basis(n_qubits)

    # --- EVE'S SIDE: pick her bases ---
    eve_bases   = quantum_random_basis(n_qubits)

    # --- BOB'S SIDE ---
    bob_bases   = quantum_random_basis(n_qubits)

    # --- QUANTUM CHANNEL: Alice -> Eve -> Bob ---
    bob_bits = []
    eve_bits = []
    for i in range(n_qubits):
        qc_alice  = alice_encode(alice_bits[i], alice_bases[i])
        qc_to_bob, eve_bit = eve_intercept(qc_alice, eve_bases[i])
        eve_bits.append(eve_bit)
        bob_bits.append(bob_measure(qc_to_bob, bob_bases[i]))

    # --- PUBLIC DISCUSSION ---
    matching_positions = [i for i in range(n_qubits) if alice_bases[i] == bob_bases[i]]
    alice_key_raw = [alice_bits[i] for i in matching_positions]
    bob_key_raw   = [bob_bits[i]   for i in matching_positions]

    # --- TAMPER CHECK ---
    n_sample = max(1, int(len(matching_positions) * sample_fraction))
    sample_idx = list(range(n_sample))

    mismatches = sum(1 for i in sample_idx if alice_key_raw[i] != bob_key_raw[i])
    mismatch_rate = mismatches / n_sample

    final_alice_key = [alice_key_raw[i] for i in range(len(alice_key_raw)) if i not in sample_idx]
    final_bob_key   = [bob_key_raw[i]   for i in range(len(bob_key_raw))   if i not in sample_idx]

    # --- REPORT ---
    print(f"Qubits sent:            {n_qubits}")
    print(f"Matching-basis bits:    {len(matching_positions)}")
    print(f"Sacrificed for check:   {n_sample}")
    print(f"Mismatches in sample:   {mismatches}  ({mismatch_rate:.1%})")
    print(f"Threshold for alarm:    {threshold:.1%}")
    print(f"Expected mismatch rate under intercept-resend: ~25%")
    if mismatch_rate > threshold:
        print(">>> ATTACK DETECTED — abort, discard the key!")
    else:
        print(">>> Channel looks clean — keep the key.  (Eve got lucky!)")
    print(f"Remaining key length:   {len(final_alice_key)}")
    print(f"Alice's key == Bob's:   {final_alice_key == final_bob_key}")
    # how much Eve learned about Alice's matching-basis bits:
    eve_correct = sum(1 for i in matching_positions if eve_bits[i] == alice_bits[i])
    print(f"Eve's accuracy on matching-basis bits: {eve_correct}/{len(matching_positions)} "
          f"({eve_correct/len(matching_positions):.1%})")

bb84_with_attacker(n_qubits=200)